# # 07 · Clustering from scratch, then the pregnancy changes

Companion to the tutorial notebook [`07_clustering_scratch_and_differential`](https://github.com/lamanno-epfl/lipidomics_tutorial_cajalcourse/tree/main/notebooks/level3).

**Kernel:** `cajal-lipidomics` · **Reads:** `06_clustered.h5ad` · **Writes:** `results/differential_overall.csv`, `results/differential_per_lipizone.csv`

One idea, used twice. First we build a clustering algorithm by hand and discover the rule that makes a clustering mean something: **a split may only exist if the two halves differ in real, measurable lipids.** That rule is a statistical test (Mann-Whitney + Benjamini-Hochberg + a fold-change bar). Then we point the very same test at the real question: **what does pregnancy do to the brain's lipids, and where?**

**The road**
- Part 1: the test, built from scratch (ranks, fold change, BH), a permutation null, then a divisive splitter gated by the test
- Part 2: control vs pregnant, overall (volcano) and per lipizone
- Part 3: reading territories through marker lipids and sorted heatmaps
- Part 4: composite scores (membrane remodeling, myelination) painted on the brain

In [ ]:
# ---- setup: the same first cell in every notebook ---------------------------------------
from lipid_project import paths
paths.ensure_dirs()
paths.summary()

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import anndata as ad
from scipy.stats import mannwhitneyu, rankdata
from statsmodels.stats.multitest import multipletests
from sklearn.decomposition import NMF
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)   # our small NMF fits are short on purpose

import cajal_lipidomics as cl
from cajal_lipidomics import analysis, plotting
from cajal_lipidomics.style import FS
cl.style.set_style()

SEED = 0
rng = np.random.default_rng(SEED)
print("ready. cajal_lipidomics", cl.__version__)

In [ ]:
adata = ad.read_h5ad(paths.stage("clustered"))
print("pixels x lipids:", adata.shape, "| lipizones:", adata.obs["lipizone"].nunique())
print("conditions:", adata.obs["Condition"].value_counts().to_dict())

# A lipid-named view on the NATIVE uMAIA scale: this is what every test below runs on.
# (A few names repeat, e.g. one lipid seen as two adducts, so make_unique adds "-1", "-2".)
lipid_view = ad.AnnData(np.asarray(adata.layers["umaia"]), obs=adata.obs.copy())
lipid_view.var_names = ad.utils.make_index_unique(pd.Index(adata.var["lipid"].astype(str)))

is_ctrl = (adata.obs["Condition"] == "naive").to_numpy()
control = lipid_view[is_ctrl].copy()
print("control pixels:", control.n_obs)

# # Part 1: the test, then a splitter built on it

## Step 1: ranks instead of values (Mann-Whitney U)

To ask "does this lipid differ between groups A and B?", the t-test compares means and assumes bell curves. MALDI intensities are skewed with long tails, so we use the **Mann-Whitney U test** (a.k.a. Wilcoxon rank-sum; same test, two names). Pool both groups, replace each value by its **rank**, and ask whether one group systematically holds the high ranks. Because it reads only order, one freak-bright pixel can't hijack it.

In [ ]:
A = np.array([1.0, 2.0, 3.0, 4.0])
B = np.array([3.5, 5.0, 6.0, 7.0])                 # B is shifted up
ranks = rankdata(np.concatenate([A, B]))           # smallest value gets rank 1 (ties would share an average rank)
nA, nB = len(A), len(B)
R_A = ranks[:nA].sum()
U_A = R_A - nA * (nA + 1) / 2                      # how many (a, b) pairs have a > b
U_B = nA * nB - U_A
print("ranks:", ranks, f"| rank-sum A = {R_A:.0f}")
print(f"U_A = {U_A:.0f}, U_B = {U_B:.0f} (they add up to nA*nB = {nA * nB})")

stat, p = mannwhitneyu(A, B, alternative="two-sided")
print(f"scipy: U = {stat:.0f} (= min(U_A, U_B)), p = {p:.4f}")

# With 4 points per group even a clean shift isn't significant. With thousands of pixels per group, a real shift gets an astronomically small p, which is also a trap: a tiny, meaningless shift gets a tiny p too.

## Step 2: fold change, the size of the difference

So every p-value gets paired with an **effect size**: **log2FC = log2(mean_B / mean_A)**. +1 means B has double, −1 half, 0 no change. Base 2 makes doublings clean integers and the volcano plot symmetric.

In [ ]:
log2fc = np.log2(B.mean() / A.mean())
print(f"mean A {A.mean():.2f}, mean B {B.mean():.2f} -> log2FC {log2fc:.2f} (B is {2 ** log2fc:.1f}x A)")

# The atlas calls a lipid **differential** only if **|log2FC| > 0.2** (about a 15% change) **and** its **adjusted** p < 0.05.

## Step 3: Benjamini-Hochberg, for many tests at once

Test 100 lipids at p < 0.05 with nothing real going on, and you still expect ~5 "hits" by chance. **Benjamini-Hochberg** controls the **false discovery rate**: among the lipids you call significant, the expected fraction of false ones is capped.
1. Sort the p-values, give them ranks i = 1..m.
2. The adjusted p (q-value) is, walking down from the largest p, the running minimum of `p(i) · m / i`.
3. Call significant everything with q < α.

In [ ]:
pvals = np.array([0.001, 0.008, 0.02, 0.04, 0.20, 0.70])
m = len(pvals)
order = np.argsort(pvals)                                  # indices that sort p ascending
sorted_p = pvals[order]
raw_q = sorted_p * m / np.arange(1, m + 1)                 # p(i) * m / i
q_sorted = np.minimum.accumulate(raw_q[::-1])[::-1].clip(max=1)   # running minimum from the top, capped at 1
q = np.empty_like(q_sorted)
q[order] = q_sorted                                        # put back in the original order

print("p:          ", pvals)
print("our q:      ", q.round(3))
print("statsmodels:", multipletests(pvals, method="fdr_bh")[1].round(3))

# Note p = 0.04 passes a naive 0.05 bar but its q climbs to 0.06: once you admit to running many tests, you need more evidence.

## The three steps fused into one gate

In [ ]:
def differential_table(X, mask_a, mask_b, min_fc=0.2, qthr=0.05, eps=1e-11):
    """Per lipid: log2FC (b/a), Mann-Whitney p, BH q, and the significance flag.

    X      : pixels x lipids
    mask_a : boolean mask of the pixels in group a (the reference)
    mask_b : boolean mask of the pixels in group b
    """
    a, b = X[mask_a], X[mask_b]                       # the two groups' rows
    # Step 2, the size: log2 of the ratio of means, per lipid. eps stops log(0) / division by 0.
    fc = np.log2((b.mean(0) + eps) / (a.mean(0) + eps))
    # Step 1, the reality check: one Mann-Whitney test per lipid column
    p = np.ones(X.shape[1])
    for j in range(X.shape[1]):
        try:
            p[j] = mannwhitneyu(a[:, j], b[:, j], alternative="two-sided").pvalue
        except ValueError:                            # all values identical: nothing to test, p stays 1
            p[j] = 1.0
    # Step 3, many tests at once: Benjamini-Hochberg across all lipids (index [1] = adjusted p-values)
    q = multipletests(p, method="fdr_bh")[1]
    # Differential = significant after correction AND a big-enough change, in either direction
    return pd.DataFrame({"log2fc": fc, "pval": p, "qval": q, "sig": (q < qthr) & (np.abs(fc) > min_fc)})


def n_differential(X, mask_a, mask_b, **kw):
    """How many lipids truly differ between two groups of pixels."""
    return int(differential_table(X, mask_a, mask_b, **kw)["sig"].sum())


print("gate ready")

# ### A permutation null for a split

Is a given cut producing more differential lipids than chance would? There's no formula for that, so we build the answer empirically. Take a real cut, **shuffle the group labels** many times (same group sizes, random membership), and count the differential lipids each time. That spread is the **null distribution**. If a shuffled cut essentially never matches the real one, the real cut carries signal.

The "real cut": split the control pixels along their strongest lipid program (high vs low), which captures the big grey/white contrast.

In [ ]:
Xc = np.asarray(control.X, float)                              # control pixels x lipids, native uMAIA scale

# A quick 6-program NMF of the control brain (shifted to be >= 0, as NMF requires)
Wc = NMF(n_components=6, init="nndsvda", random_state=SEED, max_iter=200).fit_transform(Xc - Xc.min() + 1e-7)
strongest = np.argmax(Wc.var(axis=0))                          # the program that varies most across pixels
high = Wc[:, strongest] > np.median(Wc[:, strongest])          # the real cut: above vs below its median
n_real = n_differential(Xc, high, ~high)

# The null: shuffle WHICH pixels are "high" (same 50/50 sizes) and recount, B_SHUF times
B_SHUF = 30
null = np.empty(B_SHUF, dtype=int)
for b in range(B_SHUF):
    perm = rng.permutation(high)                               # same number of True/False, random positions
    null[b] = n_differential(Xc, perm, ~perm)

# Empirical p-value: how often did a shuffle match or beat the real cut? (+1s so it's never exactly 0)
emp_p = (1 + (null >= n_real).sum()) / (1 + B_SHUF)
print(f"biology-driven cut : {n_real} differential lipids")
print(f"shuffled cuts      : {null.min()} to {null.max()} (mean {null.mean():.1f}) over {B_SHUF} shuffles")
print(f"empirical p        : {emp_p:.4f}  (the smallest possible with {B_SHUF} shuffles is {1 / (B_SHUF + 1):.4f})")

# ## The splitter: re-embed, cut in two, gate, repeat

The atlas builds lipizones **top-down**: start with all pixels, cut into two, recurse into each half, building a binary tree whose leaves are the territories. Each leaf's path of 1/2 decisions is its **barcode**. At every node:
1. **Re-embed locally**: a fresh small NMF on just this node's pixels. After the loudest contrast (grey vs white) is split off, the next most informative directions are different and local.
2. **Standardize** the programs and **cut in two** with k-means (k = 2), a readable stand-in for EUCLID's backSPIN.
3. **Gate**: keep the cut only if enough lipids differ (≥ 5 here) and both halves are big enough.
4. Accepted? Recurse into both halves. Rejected? This node is a leaf.

Two refinements the real atlas adds, named but skipped for readability: a spatial continuity check across a section stack, and an XGBoost classifier per split (you ran both inside EUCLID in notebook 06).

In [ ]:
# Scale the gate's minimum sizes to the data, so tiny datasets don't stop at the root
# 2% of the pixels, but at least 200 and at most 2000 (2000 on the real ~88k-pixel control section)
MIN_PIXELS = int(min(2000, max(200, 0.02 * len(Xc))))
barcodes = {}                                   # filled in by split_node: pixel row -> leaf barcode


def split_node(X, idx, depth, path, min_pixels=MIN_PIXELS, min_diff=5, max_depth=3, k_local=6):
    """One node of the divisive tree: re-embed -> standardize -> KMeans(2) -> gate -> recurse.

    X     : the full control lipid matrix (pixels x lipids, native uMAIA scale)
    idx   : row numbers of the pixels that belong to THIS node
    depth : how deep we are in the tree (the root is 0)
    path  : the barcode so far, e.g. "12" = took branch 1, then branch 2
    """
    label = path or "ROOT"              # the root has an empty path, so give it a readable name
    pad = "  " * depth                  # indent the printout so the tree shape is visible

    # Stop if the node is too small to split meaningfully, or we've reached the depth limit
    if len(idx) < min_pixels or depth >= max_depth:
        barcodes.update({i: label for i in idx})          # every pixel here gets this leaf's barcode
        print(f"{pad}leaf {label}: {len(idx)} px")
        return

    # 1) Re-embed locally: a small NMF on just this node's pixels.
    #    NMF needs non-negative input, so shift the values up to start just above zero.
    sub = X[idx]
    nmf = NMF(n_components=k_local, init="nndsvda", random_state=SEED, max_iter=200)
    W_loc = nmf.fit_transform(sub - sub.min() + 1e-7)

    # 2) Standardize each program (mean 0, sd 1) so no single loud program decides the cut alone,
    #    then let k-means cut the pixels into two groups (labels 0 and 1)
    lab = KMeans(n_clusters=2, random_state=SEED, n_init=10).fit_predict(StandardScaler().fit_transform(W_loc))
    a, b = lab == 0, lab == 1                              # boolean masks for the two halves

    # A cut that peels off a tiny sliver isn't a real territory split: stop here
    if min(a.sum(), b.sum()) < min_pixels // 4:
        barcodes.update({i: label for i in idx})
        print(f"{pad}leaf {label}: {len(idx)} px (unbalanced cut, stop)")
        return

    # 3) The gate: do the two halves actually differ in their lipids?
    nd = n_differential(sub, a, b)
    print(f"{pad}node {label}: {len(idx)} px -> {a.sum()}/{b.sum()}, {nd} differential lipids")
    if nd < min_diff:
        barcodes.update({i: label for i in idx})           # not enough evidence: this node stays a leaf
        print(f"{pad}  gate rejects it: keep as leaf")
        return

    # 4) Accepted: recurse into each half, appending "1" or "2" to the barcode.
    #    idx[a] picks out the original row numbers of the pixels that went into half a.
    split_node(X, idx[a], depth + 1, path + "1")
    split_node(X, idx[b], depth + 1, path + "2")


split_node(Xc, np.arange(len(Xc)), 0, "")          # start at the root with every control pixel
print(f"\n{len(set(barcodes.values()))} territories from {len(Xc)} control pixels")

# Painted on the control section with the Allen outlines on top: if the gate captured biology, colour blocks sit inside anatomical outlines rather than straddling them.

In [ ]:
obs_c = control.obs.copy()
obs_c["barcode"] = [barcodes[i] for i in range(len(obs_c))]       # look up each pixel's leaf barcode
uniq = sorted(obs_c["barcode"].unique())
cmap = dict(zip(uniq, plotting.distinct_colors(len(uniq))))       # one distinct colour per territory

fig, ax = plt.subplots(figsize=(5.8, 5.6))
# Pixel coordinates (x, -y) so the outlines from allen_contours line up exactly with the dots
ax.scatter(obs_c["x"], -obs_c["y"], c=obs_c["barcode"].map(cmap), s=4, rasterized=True)
plotting.allen_contours(obs_c, ax)                                 # black Allen region outlines on top
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("from-scratch territories (control) with Allen outlines", fontsize=FS["m"])
plt.show()
print("territories:", uniq)

# # Part 2: the same test, turned on pregnancy

Same Mann-Whitney, same BH, same fold change. Different job. Two rules:
- **Test on the uMAIA-normalized data**, never on the Harmony embedding: a batch correction can quietly erase (or invent) a condition difference.
- **Fix the direction once:** log2FC = log2(pregnant / control). Positive = higher in pregnancy.

The helper `analysis.differential_lipids` is the tested version of our gate, returning a tidy table. We check it agrees with ours, then use it.

In [ ]:
# The helper: group1 = control ("naive"), group2 = pregnant, so log2FC = log2(pregnant / control)
diff = analysis.differential_lipids(lipid_view, group_col="Condition", group1="naive", group2="pregnant")

# Our hand-built gate on the same data, same direction (mask_a = control, mask_b = pregnant)
preg_mask = (lipid_view.obs["Condition"] == "pregnant").to_numpy()
ours = differential_table(np.asarray(lipid_view.X), ~preg_mask, preg_mask)
ours.index = lipid_view.var_names
# The helper's table is sorted by fold change, so line ours up by lipid name before comparing
agree = (diff.set_index("lipid")["sig"] == ours.loc[diff["lipid"], "sig"].to_numpy()).all()
print("helper and hand-built gate agree on every lipid:", bool(agree))

print(f"\n{int(diff['sig'].sum())} of {len(diff)} lipids pass |log2FC| > 0.2 and q < 0.05")


def show(table, title):
    """Print a few rows of a result table, or '(none)' if it's empty."""
    print(f"\n{title}:")
    print(table[["lipid", "log2fc", "qval"]].to_string(index=False) if len(table) else "  (none)")


show(diff[diff["sig"] & (diff["log2fc"] > 0)].nlargest(8, "log2fc"), "top up in pregnancy")
show(diff[diff["sig"] & (diff["log2fc"] < 0)].nsmallest(6, "log2fc"), "top down in pregnancy")
diff.to_csv(paths.RESULTS / "differential_overall.csv", index=False)     # keep the full table

In [ ]:
# Do the sphingolipids (the myelin lipids) move as a class? Average their fold changes.
sph = diff["lipid"].str.startswith(("HexCer", "Cer", "SM"))
print(f"sphingolipids in the panel: {int(sph.sum())} | mean log2FC (pregnant/control): {diff.loc[sph, 'log2fc'].mean():+.3f}")

# A whole-section average pools cortex, white matter, thalamus and more into one number. If pregnancy remodels only some territories, or some up and some down, the pooled number blurs it. Hold that thought.

### The volcano plot

Each dot is a lipid: x = log2FC (left = down, right = up in pregnancy), y = −log10(q) (higher = more significant). Red dots clear both bars; the grey cloud is the unchanged majority. Then violins for the top hits, because a single fold change hides the shape of the distributions.

In [ ]:
plotting.volcano(diff, fc_thresh=0.2, q_thresh=0.05, label_col="lipid", top_n=10,
                 title=f"pregnancy vs control, all pixels ({len(diff)} lipids)")
plt.show()

# The 6 significant (q < 0.05) lipids with the biggest |log2FC|. reindex() reorders rows by |log2FC|;
# rows that weren't significant become empty (NaN) and dropna removes them.
top = diff[diff["qval"] < 0.05].reindex(diff["log2fc"].abs().sort_values(ascending=False).index)
top = top.dropna(subset=["lipid"])["lipid"].head(6).tolist()
cond = lipid_view.obs["Condition"].to_numpy()
n_panels = max(1, len(top))                                 # at least one panel, even if nothing is significant
fig, axes = plt.subplots(1, n_panels, figsize=(2.5 * n_panels, 3.4))
for ax, lip in zip(np.atleast_1d(axes), top):               # atleast_1d: works whether we got 1 axis or many
    v = np.asarray(lipid_view[:, lip].X).ravel()            # this lipid's value in every pixel
    # A violin is a mirrored, smoothed histogram: you see the whole distribution, not just its mean
    ax.violinplot([v[cond == "naive"], v[cond == "pregnant"]], showmedians=True)
    ax.set_xticks([1, 2]); ax.set_xticklabels(["control", "pregnant"], rotation=30)
    ax.set_title(lip, fontsize=FS["s"])
fig.suptitle("top pregnancy-changed lipids: full distributions", y=1.02)
plt.tight_layout()
plt.show()

# ## The real pregnancy analysis: per lipizone

Now the same test **inside each lipizone** from notebook 06. Holding tissue identity fixed, what did pregnancy change *here*? We only test lipizones with at least 200 pixels in both conditions.

In [ ]:
MIN_PX = 200                                  # minimum pixels per condition for a lipizone to be tested
rows = []                                     # one summary row per tested lipizone
lipizone_tables = {}                          # the full differential table per lipizone (reused below)
for lz in adata.obs["lipizone"].cat.categories:
    sub = lipid_view[lipid_view.obs["lipizone"] == lz]                 # just this lipizone's pixels
    n_c = int((sub.obs["Condition"] == "naive").sum())
    n_p = int((sub.obs["Condition"] == "pregnant").sum())
    if n_c < MIN_PX or n_p < MIN_PX:          # too few pixels on one side: skip rather than over-interpret
        continue
    d = analysis.differential_lipids(sub, "Condition", "naive", "pregnant")
    lipizone_tables[lz] = d
    n_up = int((d["sig"] & (d["log2fc"] > 0)).sum())                   # significant AND higher in pregnancy
    n_down = int((d["sig"] & (d["log2fc"] < 0)).sum())                 # significant AND lower in pregnancy
    rows.append((lz, n_c + n_p, n_up, n_down))

per_lz = pd.DataFrame(rows, columns=["lipizone", "n_pixels", "n_up", "n_down"]).sort_values("n_up", ascending=False)
per_lz.to_csv(paths.RESULTS / "differential_per_lipizone.csv", index=False)
print(f"{len(per_lz)} lipizones tested (>= {MIN_PX} pixels per condition)")
print(per_lz.head(12).to_string(index=False))

# A diverging bar chart: "up" counts to the right (red), "down" counts drawn as negatives to the left (blue)
top15 = per_lz.head(15)
fig, ax = plt.subplots(figsize=(7, 4.4))
yy = np.arange(len(top15))
ax.barh(yy, top15["n_up"], color="crimson", label="up in pregnancy")
ax.barh(yy, -top15["n_down"], color="steelblue", label="down in pregnancy")
ax.set_yticks(yy); ax.set_yticklabels(top15["lipizone"]); ax.invert_yaxis()
ax.axvline(0, color="k", lw=0.6)
ax.set_xlabel("number of differential lipids"); ax.set_ylabel("lipizone")
ax.set_title("where pregnancy remodels the lipidome", fontsize=FS["m"]); ax.legend(loc="lower right")
plt.show()

# The remodeling is **not uniform**: some territories move a lot, others barely. The overall test asks "does the average pixel change?"; the per-lipizone test asks "**where** does the brain change?", the biological question.

# Part 3: what each territory is made of

**Marker lipids** (one-vs-rest log2FC inside each lipizone, on the control brain) are a lipizone's molecular fingerprint, like marker genes in single-cell.

In [ ]:
ctrl_view = lipid_view[is_ctrl].copy()                     # markers are learned on the healthy control brain
# {lipizone: [(lipid, log2FC inside vs outside), ...]}, top 4 per lipizone
markers = analysis.marker_lipids(ctrl_view, group_key="lipizone", top_n=4)
for lz in list(markers)[:10]:                               # print the first 10 lipizones
    print(f"lipizone {str(lz):>3}: " + ", ".join(f"{n} ({f:+.2f})" for n, f in markers[lz]))

# Two sorted heatmaps of the control brain, the same lipids grouped two ways: by **Allen region** (anatomy) and by **lipizone** (lipid-defined). Both axes are ordered by cosine similarity, so blocks appear along the diagonal. If the two look congruent, our unsupervised lipizones recover known anatomy from molecules alone.

In [ ]:
plotting.sorted_lipid_heatmap(ctrl_view, group_key="acronym", cmap="Reds", figsize=(14, 8),
                              title="control: Allen region x lipid")
plt.show()
plotting.sorted_lipid_heatmap(ctrl_view, group_key="lipizone", cmap="Reds", figsize=(14, 6),
                              title="control: lipizone x lipid")
plt.show()

# # Part 4: composite scores painted on the brain

## Membrane remodeling: net building or net turnover

The atlas summarises a territory with one number: the **sum of all its lipid log2FCs**. Mostly up → positive → net **building** of membrane. Mostly down → net **turnover**. We paint it on the pregnant section with a diverging map centred at 0 (red = building, blue = turnover).

In [ ]:
# One number per lipizone: the sum of its lipids' log2FCs (from the per-lipizone tables above)
membrane = pd.Series(analysis.membrane_remodeling_score(lipizone_tables)).sort_values()
print("most net-turnover lipizones:\n" + membrane.head(5).round(2).to_string())
print("\nmost net-building lipizones:\n" + membrane.tail(5).round(2).to_string())

preg = adata[~is_ctrl].copy()
# Give each pregnant pixel its lipizone's score (.map looks each lipizone up in the `membrane` Series)
preg.obs["membrane"] = preg.obs["lipizone"].astype(str).map(membrane).astype(float)
shown = preg.obs.dropna(subset=["membrane"])               # untested lipizones have no score: leave them out
# Symmetric colour limits around 0, so white means "no net change" (the "or 1.0" guards an all-zero case)
vmax = float(np.nanmax(np.abs(shown["membrane"]))) or 1.0
fig, ax = plt.subplots(figsize=(6, 5.8))
sc = ax.scatter(shown["x"], -shown["y"], c=shown["membrane"], cmap="RdBu_r", vmin=-vmax, vmax=vmax, s=4, rasterized=True)
plotting.allen_contours(preg.obs, ax)
ax.set_aspect("equal"); ax.axis("off")
ax.set_title("membrane remodeling per lipizone (pregnant section)", fontsize=FS["m"])
cl.style.lightweight_colorbar(sc, ax, label="sum of lipid log2FC")
plt.show()

# ## Myelination: where the sphingolipids live

Per pixel: the mean **z-scored** intensity of all sphingolipids (HexCer, Cer, SM). Z-scoring first puts each on the same footing so no single bright channel dominates. It should trace the white-matter tracts, which validates it as a myelin readout.

In [ ]:
# Mean z-scored sphingolipid intensity per pixel (read analysis.myelination_score: it's 5 lines)
adata.obs["myelination"] = analysis.myelination_score(lipid_view)
mye = adata.obs["myelination"].to_numpy()

vmin, vmax = np.quantile(mye, [0.02, 0.98])                 # one shared colour scale for both sections
fig, axes = plt.subplots(1, 2, figsize=(10.5, 5), constrained_layout=True)
for ax, c in zip(axes, ["naive", "pregnant"]):
    s = adata.obs[adata.obs["Condition"] == c]
    sc = ax.scatter(s["x"], -s["y"], c=s["myelination"], cmap="magma", vmin=vmin, vmax=vmax, s=4, rasterized=True)
    plotting.allen_contours(s, ax, color="white", lw=0.3, s=0.4)
    ax.set_aspect("equal"); ax.axis("off"); ax.set_title(f"myelination score: {c}", fontsize=FS["m"])
fig.colorbar(sc, ax=axes, shrink=0.6, label="mean z-scored sphingolipid")
plt.show()

# Now the question the pregnancy story rests on: does that score rise, and **where**? Three views of the same score:

In [ ]:
wm = adata.obs["allencolor"].astype(str).str.lower().eq("#cccccc").to_numpy()    # white matter, the atlas's own definition

print(f"whole section : control {mye[is_ctrl].mean():+.3f} -> pregnant {mye[~is_ctrl].mean():+.3f}")
print(f"white matter  : control {mye[is_ctrl & wm].mean():+.3f} -> pregnant {mye[~is_ctrl & wm].mean():+.3f}"
      f"   ({int((is_ctrl & wm).sum())} vs {int((~is_ctrl & wm).sum())} pixels)")

up = tested = 0
# groupby(...).indices gives {lipizone: array of row numbers}, so we can index numpy arrays directly
for lz, ix in adata.obs.groupby("lipizone", observed=True).indices.items():
    c = is_ctrl[ix]                                          # which of this lipizone's pixels are control
    if c.sum() < MIN_PX or (~c).sum() < MIN_PX:
        continue
    tested += 1
    up += int(mye[ix][~c].mean() > mye[ix][c].mean())       # +1 if the pregnant mean is higher
print(f"per lipizone  : {up} of {tested} lipizones raise their myelination score in pregnancy")

# How to read the three together:
- **Whole section:** the pooled average is dominated by how much grey vs white matter each slice happens to contain, so a real change can get swamped.
- **White matter only:** restrict to where myelin actually lives. This is where the paper's headline (myelination signal up in pregnant white matter) should show up.
- **Per lipizone:** check whether the territories that rise are the sphingolipid-rich, white-matter-like ones.

If white matter rises while the whole-section number barely moves, that's the dilution effect in action. Whatever you see, remember the caveat: one section per condition, so this is a direction, not a proven effect size.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
plotting.tsne_colored(adata, by="myelination", kind="value", ax=axes[0], title="t-SNE by myelination score")
# tsne_colored(kind="color") reads a hex colour per pixel from obs, so make one from the condition
adata.obs["cond_color"] = adata.obs["Condition"].map({"naive": "#3366cc", "pregnant": "#dd3333"})
plotting.tsne_colored(adata, by="cond_color", kind="color", ax=axes[1], title="t-SNE by condition (blue control, red pregnant)")
plt.tight_layout()
plt.show()

# ## Recap

- A clustering means something only if its splits are backed by **differential lipids**: random cuts get ~0, biological cuts get many (permutation null).
- The test: **Mann-Whitney** (ranks) + **log2FC** (size) + **Benjamini-Hochberg** (many tests).
- On pregnancy, the whole-section verdict is blunt; **per lipizone** and **in white matter** is where to look.
- **Marker lipids** and sorted heatmaps say what each territory is; **composite scores** summarise remodeling and myelination.
- Tables saved to `results/differential_overall.csv` and `results/differential_per_lipizone.csv`.

**Next (08):** which genes explain these changes?